## Model Store Setup

The first step is to set up the SageMaker environment that will be used to train and register the model. I will use the execution role associated with the SageMaker environment and create a SageMaker session.

I will also identify the AWS region and default S3 bucket. The S3 bucket will be used to store the training data and the model artifacts created during the training job.


In [1]:
# Set up the SageMaker environment
import boto3
import sagemaker
try:
    # Get the execution role used by the SageMaker environment
    role = sagemaker.get_execution_role()
    # Create the SageMaker session
    sess = sagemaker.Session()
    # Get the AWS region and default S3 bucket
    region = sess.boto_region_name
    bucket = sess.default_bucket()
    # S3 location used for the Week 4 model files
    prefix = "week4-breast-cancer-xgboost"
    print("SageMaker environment set up successfully.")
    print("\nAWS region:")
    print(region)
    print("\nS3 bucket:")
    print(bucket)
    print("\nS3 prefix:")
    print(prefix)
    print("\nSageMaker execution role:")
    print(role)
except Exception as e:
    print(f"SageMaker environment setup failed: {e}")
    raise

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


SageMaker environment set up successfully.

AWS region:
us-east-1

S3 bucket:
sagemaker-us-east-1-614554462636

S3 prefix:
week4-breast-cancer-xgboost

SageMaker execution role:
arn:aws:iam::614554462636:role/LabRole


## Load the Breast Cancer Dataset

The model will use the Breast Cancer Wisconsin Diagnostic dataset from the Week 4 lab. The dataset contains information about characteristics of cell nuclei collected from breast mass images and will be used to train a model that predicts whether a tumor is malignant or benign.

I will download the dataset from the SageMaker example files S3 bucket and load it into a pandas DataFrame. I will then assign the column names provided with the dataset and inspect its shape and a sample of the records.


In [2]:
# Load the breast cancer dataset
import pandas as pd
try:
    # Create the S3 client
    s3 = boto3.client("s3")
    # Download the dataset used in the Week 4 lab
    filename = "wdbc.csv"
    source_bucket = f"sagemaker-example-files-prod-{region}"
    source_key = "datasets/tabular/breast_cancer/wdbc.csv"
    s3.download_file(
        source_bucket,
        source_key,
        filename
    )
    # Load the dataset
    data = pd.read_csv(
        filename,
        header=None
    )
    # Add the column names provided with the dataset
    data.columns = [
        "id",
        "diagnosis",
        "radius_mean",
        "texture_mean",
        "perimeter_mean",
        "area_mean",
        "smoothness_mean",
        "compactness_mean",
        "concavity_mean",
        "concave points_mean",
        "symmetry_mean",
        "fractal_dimension_mean",
        "radius_se",
        "texture_se",
        "perimeter_se",
        "area_se",
        "smoothness_se",
        "compactness_se",
        "concavity_se",
        "concave points_se",
        "symmetry_se",
        "fractal_dimension_se",
        "radius_worst",
        "texture_worst",
        "perimeter_worst",
        "area_worst",
        "smoothness_worst",
        "compactness_worst",
        "concavity_worst",
        "concave points_worst",
        "symmetry_worst",
        "fractal_dimension_worst"
    ]
    print("Breast cancer dataset loaded successfully.")
    print("\nDataset shape:")
    print(data.shape)
    print("\nDataset columns:")
    print(list(data.columns))
    print("\nSample records:")
    display(data.head())
except Exception as e:
    print(f"Breast cancer dataset loading failed: {e}")
    raise

Breast cancer dataset loaded successfully.

Dataset shape:
(569, 32)

Dataset columns:
['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst']

Sample records:


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## Prepare the Target Variable

The diagnosis field contains the values M for malignant and B for benign. Since the XGBoost model will perform binary classification, these values need to be converted into numeric values before training.

I will convert malignant diagnoses to 1 and benign diagnoses to 0. I will also check the class distribution and missing values to make sure the dataset is ready to be split into training and validation sets.


In [3]:
# Prepare the diagnosis field for binary classification
try:
    # Check the diagnosis values before conversion
    print("Diagnosis values before conversion:")
    print(data["diagnosis"].value_counts())
    # Convert malignant to 1 and benign to 0
    data["diagnosis"] = data["diagnosis"].apply(
        lambda x: ((x == "M")) + 0
    )
    print("\nDiagnosis values after conversion:")
    print(data["diagnosis"].value_counts())
    # Check for missing values in the dataset
    print("\nMissing values in the dataset:")
    print(data.isnull().sum().sum())
    print("\nSample prepared records:")
    display(
        data[
            [
                "id",
                "diagnosis",
                "radius_mean",
                "texture_mean",
                "perimeter_mean",
                "area_mean"
            ]
        ].head()
    )
except Exception as e:
    print(f"Diagnosis preparation failed: {e}")
    raise

Diagnosis values before conversion:
diagnosis
B    357
M    212
Name: count, dtype: int64

Diagnosis values after conversion:
diagnosis
0    357
1    212
Name: count, dtype: int64

Missing values in the dataset:
0

Sample prepared records:


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean
0,842302,1,17.99,10.38,122.80,1001.0
1,842517,1,20.57,17.77,132.90,1326.0
2,84300903,1,19.69,21.25,130.00,1203.0
3,84348301,1,11.42,20.38,77.58,386.1
4,84358402,1,20.29,14.34,135.10,1297.0


### Target Variable Preparation Results

The diagnosis field was successfully converted into the numeric values required for binary classification. The dataset contains 357 benign records represented by 0 and 212 malignant records represented by 1.

There are no missing values in the dataset, so no additional missing value handling is required before training. The next step is to split the data into training, validation, and batch inference sets.


In [4]:
# Split the dataset for training, validation, and batch inference
import numpy as np
try:
    # Create the random split used for the three datasets
    rand_split = np.random.rand(len(data))
    train_list = rand_split < 0.8
    val_list = (
        (rand_split >= 0.8)
        & (rand_split < 0.9)
    )
    batch_list = rand_split >= 0.9
    # Remove the ID field from the training and validation data
    data_train = data[
        train_list
    ].drop(
        ["id"],
        axis=1
    )
    data_val = data[
        val_list
    ].drop(
        ["id"],
        axis=1
    )
    # Keep the ID for the batch data but remove the target
    data_batch = data[
        batch_list
    ].drop(
        ["diagnosis"],
        axis=1
    )
    # Create a batch copy without the ID for model input
    data_batch_noID = data_batch.drop(
        ["id"],
        axis=1
    )
    print("Dataset split completed successfully.")
    print("\nTraining records:")
    print(len(data_train))
    print("\nValidation records:")
    print(len(data_val))
    print("\nBatch inference records:")
    print(len(data_batch))
    print("\nTraining data shape:")
    print(data_train.shape)
    print("\nValidation data shape:")
    print(data_val.shape)
    print("\nBatch data shape:")
    print(data_batch.shape)
    print("\nBatch data without ID shape:")
    print(data_batch_noID.shape)
except Exception as e:
    print(f"Dataset split failed: {e}")
    raise

Dataset split completed successfully.

Training records:
438

Validation records:
69

Batch inference records:
62

Training data shape:
(438, 31)

Validation data shape:
(69, 31)

Batch data shape:
(62, 31)

Batch data without ID shape:
(62, 30)


### Training and Validation Data Split

The dataset was split into 438 training records, 69 validation records, and 62 records for batch inference. The training and validation datasets contain the diagnosis field followed by the 30 input features used by the model.

The ID field was removed from the training and validation data because it is not a predictive feature. A second version of the batch data was also created without the ID field so that its input features match the features used during model training.

The next step is to save these datasets as CSV files and upload them to S3 so they can be used by the SageMaker training job.


In [5]:
# Save the datasets and upload them to S3
try:
    # Save the training data without headers or an index
    train_file = "train_data.csv"
    data_train.to_csv(
        train_file,
        index=False,
        header=False
    )
    train_s3_path = sess.upload_data(
        train_file,
        key_prefix=f"{prefix}/train"
    )
    # Save the validation data
    validation_file = "validation_data.csv"
    data_val.to_csv(
        validation_file,
        index=False,
        header=False
    )
    validation_s3_path = sess.upload_data(
        validation_file,
        key_prefix=f"{prefix}/validation"
    )
    # Save the batch data with the ID field
    batch_file = "batch_data.csv"
    data_batch.to_csv(
        batch_file,
        index=False,
        header=False
    )
    batch_s3_path = sess.upload_data(
        batch_file,
        key_prefix=f"{prefix}/batch"
    )
    # Save the batch data without the ID field
    batch_file_noID = "batch_data_noID.csv"
    data_batch_noID.to_csv(
        batch_file_noID,
        index=False,
        header=False
    )
    batch_noID_s3_path = sess.upload_data(
        batch_file_noID,
        key_prefix=f"{prefix}/batch"
    )
    print("Datasets uploaded to S3 successfully.")
    print("\nTraining data:")
    print(train_s3_path)
    print("\nValidation data:")
    print(validation_s3_path)
    print("\nBatch data:")
    print(batch_s3_path)
    print("\nBatch data without ID:")
    print(batch_noID_s3_path)
except Exception as e:
    print(f"Dataset upload failed: {e}")
    raise

Datasets uploaded to S3 successfully.

Training data:
s3://sagemaker-us-east-1-614554462636/week4-breast-cancer-xgboost/train/train_data.csv

Validation data:
s3://sagemaker-us-east-1-614554462636/week4-breast-cancer-xgboost/validation/validation_data.csv

Batch data:
s3://sagemaker-us-east-1-614554462636/week4-breast-cancer-xgboost/batch/batch_data.csv

Batch data without ID:
s3://sagemaker-us-east-1-614554462636/week4-breast-cancer-xgboost/batch/batch_data_noID.csv


### S3 Data Upload Results

The training, validation, and batch inference datasets were successfully uploaded to the S3 bucket. Separate S3 locations are available for the training and validation data that will be passed to the SageMaker training job.

The batch data was uploaded both with and without the ID field. The version without the ID contains the same 30 input features used during training and can be used later for inference.

The next step is to configure the SageMaker XGBoost estimator and define the hyperparameters that will be used to train the binary classification model.


In [6]:
# Configure the SageMaker XGBoost estimator
from time import gmtime, strftime
try:
    # Create a unique name for the training job
    job_name = (
        "xgb-" +
        strftime("%Y-%m-%d-%H-%M-%S", gmtime())
    )
    # Set the S3 location for the model output
    output_location = (
        f"s3://{bucket}/{prefix}/output/{job_name}"
    )
    # Get the SageMaker XGBoost container image
    image = sagemaker.image_uris.retrieve(
        framework="xgboost",
        region=region,
        version="1.7-1"
    )
    # Create the XGBoost estimator
    sm_estimator = sagemaker.estimator.Estimator(
        image,
        role,
        instance_count=1,
        instance_type="ml.m5.xlarge",
        volume_size=50,
        input_mode="File",
        output_path=output_location,
        sagemaker_session=sess
    )
    # Set the hyperparameters 
    sm_estimator.set_hyperparameters(
        objective="binary:logistic",
        max_depth=5,
        eta=0.2,
        gamma=4,
        min_child_weight=6,
        subsample=0.8,
        verbosity=0,
        num_round=100
    )
    print("XGBoost estimator configured successfully.")
    print("\nTraining job name:")
    print(job_name)
    print("\nXGBoost image:")
    print(image)
    print("\nModel output location:")
    print(output_location)
    print("\nModel hyperparameters:")
    print(sm_estimator.hyperparameters())
except Exception as e:
    print(f"XGBoost estimator configuration failed: {e}")
    raise

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/estimator.py:588: SageMakerV2DeprecationWarning: Estimator is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
Estimator is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


XGBoost estimator configured successfully.

Training job name:
xgb-2026-09-25-17-57-16

XGBoost image:
683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1

Model output location:
s3://sagemaker-us-east-1-614554462636/week4-breast-cancer-xgboost/output/xgb-2026-09-25-17-57-16

Model hyperparameters:
{'objective': 'binary:logistic', 'max_depth': 5, 'eta': 0.2, 'gamma': 4, 'min_child_weight': 6, 'subsample': 0.8, 'verbosity': 0, 'num_round': 100}


### XGBoost Model Configuration

The SageMaker XGBoost estimator was configured using version 1.7-1 and an ml.m5.xlarge training instance. The model uses the binary:logistic objective because the goal is to predict whether a tumor is malignant or benign.

The training configuration uses a maximum tree depth of 5, learning rate of 0.2, gamma of 4, minimum child weight of 6, subsample rate of 0.8, and 100 boosting rounds. The trained model artifacts will be stored in the S3 output location created for this training job.

The next step is to connect the training and validation data from S3 and start the SageMaker training job.


In [7]:
# Train the XGBoost model using the S3 datasets
try:
    # Create the training data input
    train_data = sagemaker.inputs.TrainingInput(
        f"s3://{bucket}/{prefix}/train",
        distribution="FullyReplicated",
        content_type="text/csv",
        s3_data_type="S3Prefix"
    )
    # Create the validation data input
    validation_data = sagemaker.inputs.TrainingInput(
        f"s3://{bucket}/{prefix}/validation",
        distribution="FullyReplicated",
        content_type="text/csv",
        s3_data_type="S3Prefix"
    )
    # Add the datasets to the training channels
    data_channels = {
        "train": train_data,
        "validation": validation_data
    }
    print("Training and validation channels prepared successfully.")
    print("\nStarting SageMaker training job:")
    print(job_name)
    # Start the SageMaker training job
    sm_estimator.fit(
        inputs=data_channels,
        job_name=job_name,
        logs=True
    )
    print("\nModel training completed successfully.")
except Exception as e:
    print(f"XGBoost model training failed: {e}")
    raise

INFO:sagemaker:Creating training-job with name: xgb-2026-09-25-17-57-16


Training and validation channels prepared successfully.

Starting SageMaker training job:
xgb-2026-09-25-17-57-16
2026-09-25 17:59:24 Starting - Starting the training job...
2026-09-25 17:59:39 Starting - Preparing the instances for training...
2026-09-25 18:00:05 Downloading - Downloading input data...
2026-09-25 18:00:35 Downloading - Downloading the training image...
2026-09-25 18:01:15 Training - Training image download completed. Training in progress../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-09-25 18:01:25.364 ip-10-0-208-95.ec2.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-09-25 18:01:25.436 ip-10-0-208-95.ec2.internal:7 INFO profiler_config_parser.py:111

### Model Training Results

The SageMaker XGBoost training job completed successfully using 438 training records with 30 input features and 69 validation records. The model completed all 100 boosting rounds and SageMaker uploaded the trained model artifact to S3.

The training and validation log loss decreased during training, with the final training log loss at approximately 0.077 and validation log loss at approximately 0.099. The next step is to inspect the completed training job and retrieve the S3 location of the trained model artifact. This information will be used when the model is added to SageMaker Model Registry.


In [8]:
# Inspect the completed SageMaker training job
try:
    # Create the SageMaker client
    sagemaker_client = boto3.client(
        "sagemaker",
        region_name=region
    )
    # Get the details for the completed training job
    training_job_details = (
        sagemaker_client.describe_training_job(
            TrainingJobName=job_name
        )
    )
    # Get the S3 location of the trained model artifact
    model_artifact = training_job_details[
        "ModelArtifacts"
    ]["S3ModelArtifacts"]
    print("Training job details retrieved successfully.")
    print("\nTraining job name:")
    print(training_job_details["TrainingJobName"])
    print("\nTraining job status:")
    print(training_job_details["TrainingJobStatus"])
    print("\nTraining instance type:")
    print(
        training_job_details[
            "ResourceConfig"
        ]["InstanceType"]
    )
    print("\nTraining image:")
    print(
        training_job_details[
            "AlgorithmSpecification"
        ]["TrainingImage"]
    )
    print("\nModel artifact:")
    print(model_artifact)
except Exception as e:
    print(f"Training job inspection failed: {e}")
    raise

Training job details retrieved successfully.

Training job name:
xgb-2026-09-25-17-57-16

Training job status:
Completed

Training instance type:
ml.m5.xlarge

Training image:
683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1

Model artifact:
s3://sagemaker-us-east-1-614554462636/week4-breast-cancer-xgboost/output/xgb-2026-09-25-17-57-16/xgb-2026-09-25-17-57-16/output/model.tar.gz


## Part 1: Set Up Model Group

The Model Group will be used to organize different versions of the breast cancer classification model. New model packages can be added to this group when the model algorithm, training data, input features, or hyperparameters are changed.

I will create a Model Group with an informative name and a short description of its purpose. Before creating the group, I will check whether it already exists so the notebook can be run again without creating a duplicate resource.


In [9]:
# Create the Model Group for the breast cancer model
model_package_group_name = (
    "xgboost-breast-cancer-detection"
)
model_package_group_description = (
    "XGBoost models used to classify breast tumors "
    "as malignant or benign using diagnostic features."
)
try:
    # Check whether the Model Group already exists
    try:
        existing_group = (
            sagemaker_client.describe_model_package_group(
                ModelPackageGroupName=model_package_group_name
            )
        )
        model_package_group_arn = (
            existing_group["ModelPackageGroupArn"]
        )
        print("Model Group already exists.")
        print("\nModel Group ARN:")
        print(model_package_group_arn)
    except sagemaker_client.exceptions.ClientError as e:
        error_code = e.response["Error"]["Code"]
        if error_code != "ValidationException":
            raise
        # Create the Model Group
        response = (
            sagemaker_client.create_model_package_group(
                ModelPackageGroupName=(
                    model_package_group_name
                ),
                ModelPackageGroupDescription=(
                    model_package_group_description
                )
            )
        )
        model_package_group_arn = (
            response["ModelPackageGroupArn"]
        )
        print("Model Group created successfully.")
        print("\nModel Group ARN:")
        print(model_package_group_arn)
except Exception as e:
    print(f"Model Group setup failed: {e}")
    raise

Model Group created successfully.

Model Group ARN:
arn:aws:sagemaker:us-east-1:614554462636:model-package-group/xgboost-breast-cancer-detection


In [10]:
# Describe the Model Group for verification
try:
    # Get the Model Group details from SageMaker
    model_group_details = (
        sagemaker_client.describe_model_package_group(
            ModelPackageGroupName=model_package_group_name
        )
    )
    print("Model Group details retrieved successfully.")
    print("\nModel Group name:")
    print(model_group_details["ModelPackageGroupName"])
    print("\nModel Group ARN:")
    print(model_group_details["ModelPackageGroupArn"])
    print("\nModel Group description:")
    print(model_group_details["ModelPackageGroupDescription"])
    print("\nModel Group status:")
    print(model_group_details["ModelPackageGroupStatus"])
    print("\nCreation time:")
    print(model_group_details["CreationTime"])
except Exception as e:
    print(f"Model Group description failed: {e}")
    raise

Model Group details retrieved successfully.

Model Group name:
xgboost-breast-cancer-detection

Model Group ARN:
arn:aws:sagemaker:us-east-1:614554462636:model-package-group/xgboost-breast-cancer-detection

Model Group description:
XGBoost models used to classify breast tumors as malignant or benign using diagnostic features.

Model Group status:
Completed

Creation time:
2026-09-25 18:06:22.161000+00:00


### Model Group Results

The Model Group was created successfully with the name `xgboost-breast-cancer-detection`. The group will be used to track versions of the XGBoost breast cancer classification model as changes are made to the model, training data, features, or hyperparameters.

The Model Group status is Completed, confirming that it is available in SageMaker Model Registry. The next step is to create the first Model Package using the trained model artifact and XGBoost inference image.


## Part 2: Set Up Model Package

The Model Package will represent the current version of the trained XGBoost model. It will connect the trained model artifact stored in S3 with the XGBoost container image required to run inference.

The inference specification will also document the supported input and output content types. This model accepts CSV input containing the 30 diagnostic features used during training and returns prediction results in CSV format.

The model package will be added to the Model Group created in the previous step so future versions of the model can be tracked in the same group.


In [11]:
# Create the first Model Package for the trained model
try:
    # Define the inference specification
    inference_specification = {
        "Containers": [
            {
                "Image": image,
                "ModelDataUrl": model_artifact
            }
        ],
        "SupportedContentTypes": [
            "text/csv"
        ],
        "SupportedResponseMIMETypes": [
            "text/csv"
        ]
    }
    # Register the trained model in the Model Group
    model_package_response = (
        sagemaker_client.create_model_package(
            ModelPackageGroupName=model_package_group_name,
            ModelPackageDescription=(
                "XGBoost breast cancer classifier trained "
                "using 30 diagnostic features."
            ),
            InferenceSpecification=inference_specification,
            ModelApprovalStatus="PendingManualApproval"
        )
    )
    # Save the ARN for the registered model version
    model_package_arn = (
        model_package_response["ModelPackageArn"]
    )
    print("Model Package created successfully.")
    print("\nModel Package ARN:")
    print(model_package_arn)
    print("\nModel approval status:")
    print("PendingManualApproval")
except Exception as e:
    print(f"Model Package creation failed: {e}")
    raise

Model Package created successfully.

Model Package ARN:
arn:aws:sagemaker:us-east-1:614554462636:model-package/xgboost-breast-cancer-detection/1

Model approval status:
PendingManualApproval


In [13]:
# Describe the Model Package for verification
try:
    # Get the registered Model Package details
    model_package_details = (
        sagemaker_client.describe_model_package(
            ModelPackageName=model_package_arn
        )
    )
    # Get the inference specification
    inference_details = (
        model_package_details["InferenceSpecification"]
    )
    container_details = (
        inference_details["Containers"][0]
    )
    print("Model Package details retrieved successfully.")
    print("\nModel Package ARN:")
    print(model_package_details["ModelPackageArn"])
    print("\nModel Package version:")
    print(model_package_details["ModelPackageVersion"])
    print("\nModel Package description:")
    print(model_package_details["ModelPackageDescription"])
    print("\nModel Package status:")
    print(model_package_details["ModelPackageStatus"])
    print("\nModel approval status:")
    print(model_package_details["ModelApprovalStatus"])
    print("\nInference image:")
    print(container_details["Image"])
    print("\nModel data artifact:")
    print(container_details["ModelDataUrl"])
    print("\nSupported input content types:")
    print(inference_details["SupportedContentTypes"])
    print("\nSupported response MIME types:")
    print(inference_details["SupportedResponseMIMETypes"])
    print("\nCreation time:")
    print(model_package_details["CreationTime"])
except Exception as e:
    print(f"Model Package description failed: {e}")
    raise

Model Package details retrieved successfully.

Model Package ARN:
arn:aws:sagemaker:us-east-1:614554462636:model-package/xgboost-breast-cancer-detection/1

Model Package version:
1

Model Package description:
XGBoost breast cancer classifier trained using 30 diagnostic features.

Model Package status:
Completed

Model approval status:
PendingManualApproval

Inference image:
683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1

Model data artifact:
s3://sagemaker-us-east-1-614554462636/week4-breast-cancer-xgboost/output/xgb-2026-09-25-17-57-16/xgb-2026-09-25-17-57-16/output/model.tar.gz

Supported input content types:
['text/csv']

Supported response MIME types:
['text/csv']

Creation time:
2026-09-25 18:11:37.101000+00:00


### Model Package Results

Model Package version 1 was successfully registered in the `xgboost-breast-cancer-detection` Model Group. The package contains the XGBoost 1.7-1 inference image and the trained `model.tar.gz` artifact stored in S3.

The package supports CSV input and output and currently has a PendingManualApproval status. Future versions of the model can be registered in the same Model Group when changes are made to the model, training data, features, or hyperparameters.

The next step is to create a Model Card that documents the purpose, training process, input features, hyperparameters, intended use, and evaluation details for this model version.


## Part 3: Write the Model Card

The Model Card will document the important details about the current version of the breast cancer classification model. This includes the model algorithm, owner, purpose, intended use, training process, hyperparameters, input features, and evaluation results.

The Model Card will also reference the registered Model Package and trained model artifact so the documentation can be connected back to the model stored in SageMaker Model Registry.


In [14]:
# Prepare the content for the Model Card
import json
try:
    # Store the input features used to train the model
    input_features = [
        "radius_mean",
        "texture_mean",
        "perimeter_mean",
        "area_mean",
        "smoothness_mean",
        "compactness_mean",
        "concavity_mean",
        "concave points_mean",
        "symmetry_mean",
        "fractal_dimension_mean",
        "radius_se",
        "texture_se",
        "perimeter_se",
        "area_se",
        "smoothness_se",
        "compactness_se",
        "concavity_se",
        "concave points_se",
        "symmetry_se",
        "fractal_dimension_se",
        "radius_worst",
        "texture_worst",
        "perimeter_worst",
        "area_worst",
        "smoothness_worst",
        "compactness_worst",
        "concavity_worst",
        "concave points_worst",
        "symmetry_worst",
        "fractal_dimension_worst"
    ]
    # Store the hyperparameters used during training
    model_hyperparameters = {
        "objective": "binary:logistic",
        "max_depth": 5,
        "eta": 0.2,
        "gamma": 4,
        "min_child_weight": 6,
        "subsample": 0.8,
        "verbosity": 0,
        "num_round": 100
    }
    # Create the Model Card content
    model_card_content = {
        "model_overview": {
            "model_description": (
                "XGBoost binary classification model used to "
                "classify breast tumors as malignant or benign."
            ),
            "model_creator": "Student Model Developer",
            "model_owner": "Student Model Developer",
            "model_artifact": [
                model_artifact
            ],
            "algorithm_type": "XGBoost 1.7-1",
            "problem_type": "Binary Classification"
        },
        "intended_uses": {
            "purpose_of_model": (
                "Classify breast tumor observations as malignant "
                "or benign using diagnostic measurements."
            ),
            "intended_uses": (
                "Educational demonstration of binary classification "
                "and SageMaker model governance."
            ),
            "factors_affecting_model_efficiency": (
                "Model performance depends on the quality and "
                "representativeness of the diagnostic input features."
            ),
            "risk_rating": "unknown",
            "explanations_for_risk_rating": (
                "This model was created for educational purposes "
                "and has not been validated for clinical use."
            )
        },
        "business_details": {
            "business_problem": (
                "Demonstrate how diagnostic measurements can be "
                "used to classify breast tumor observations."
            ),
            "business_stakeholders": (
                "Student model developer and course reviewers."
            ),
            "line_of_business": (
                "Machine learning education and model governance."
            )
        },
        "training_details": {
            "training_observations": (
                "The model was trained with 438 records and "
                "validated with 69 records using 30 input features. "
                "Training completed after 100 boosting rounds."
            )
        },
        "additional_information": {
            "custom_details": {
                "training_job_name": job_name,
                "training_instance": "ml.m5.xlarge",
                "training_image": image,
                "model_package_arn": model_package_arn,
                "model_package_group": model_package_group_name,
                "training_records": "438",
                "validation_records": "69",
                "training_logloss": "0.07712",
                "validation_logloss": "0.09882",
                "input_features": ", ".join(input_features),
                "hyperparameters": json.dumps(
                    model_hyperparameters
                )
            }
        }
    }
    # Convert the Model Card content to a JSON string
    model_card_json = json.dumps(
        model_card_content,
        indent=2
    )
    print("Model Card content prepared successfully.")
    print("\nModel owner:")
    print(
        model_card_content[
            "model_overview"
        ]["model_owner"]
    )
    print("\nAlgorithm:")
    print(
        model_card_content[
            "model_overview"
        ]["algorithm_type"]
    )
    print("\nNumber of input features:")
    print(len(input_features))
    print("\nHyperparameters:")
    print(model_hyperparameters)
    print("\nEvaluation results:")
    print("Training log loss: 0.07712")
    print("Validation log loss: 0.09882")
except Exception as e:
    print(f"Model Card content preparation failed: {e}")
    raise

Model Card content prepared successfully.

Model owner:
Student Model Developer

Algorithm:
XGBoost 1.7-1

Number of input features:
30

Hyperparameters:
{'objective': 'binary:logistic', 'max_depth': 5, 'eta': 0.2, 'gamma': 4, 'min_child_weight': 6, 'subsample': 0.8, 'verbosity': 0, 'num_round': 100}

Evaluation results:
Training log loss: 0.07712
Validation log loss: 0.09882


In [17]:
# Create the SageMaker Model Card
model_card_name = "xgboost-breast-cancer-model-card"
try:
    # Update the risk rating to the value expected by AWS
    model_card_content[
        "intended_uses"
    ]["risk_rating"] = "Unknown"
    # Convert the Model Card content to a single line JSON string
    model_card_json = json.dumps(
        model_card_content
    )
    # Create the Model Card
    model_card_response = (
        sagemaker_client.create_model_card(
            ModelCardName=model_card_name,
            SecurityConfig={},
            Content=model_card_json,
            ModelCardStatus="Draft"
        )
    )
    # Save the Model Card ARN
    model_card_arn = (
        model_card_response["ModelCardArn"]
    )
    print("Model Card created successfully.")
    print("\nModel Card name:")
    print(model_card_name)
    print("\nModel Card ARN:")
    print(model_card_arn)
    print("\nModel Card status:")
    print("Draft")
except Exception as e:
    print(f"Model Card creation failed: {e}")
    raise

Model Card created successfully.

Model Card name:
xgboost-breast-cancer-model-card

Model Card ARN:
arn:aws:sagemaker:us-east-1:614554462636:model-card/xgboost-breast-cancer-model-card

Model Card status:
Draft


In [18]:
# Describe the Model Card for verification
try:
    # Get the Model Card details from SageMaker
    model_card_details = (
        sagemaker_client.describe_model_card(
            ModelCardName=model_card_name
        )
    )
    # Convert the stored Model Card content back to a dictionary
    stored_card_content = json.loads(
        model_card_details["Content"]
    )
    print("Model Card details retrieved successfully.")
    print("\nModel Card name:")
    print(model_card_details["ModelCardName"])
    print("\nModel Card ARN:")
    print(model_card_details["ModelCardArn"])
    print("\nModel Card version:")
    print(model_card_details["ModelCardVersion"])
    print("\nModel Card status:")
    print(model_card_details["ModelCardStatus"])
    print("\nModel owner:")
    print(
        stored_card_content[
            "model_overview"
        ]["model_owner"]
    )
    print("\nAlgorithm:")
    print(
        stored_card_content[
            "model_overview"
        ]["algorithm_type"]
    )
    print("\nProblem type:")
    print(
        stored_card_content[
            "model_overview"
        ]["problem_type"]
    )
    print("\nPurpose of model:")
    print(
        stored_card_content[
            "intended_uses"
        ]["purpose_of_model"]
    )
    print("\nIntended use:")
    print(
        stored_card_content[
            "intended_uses"
        ]["intended_uses"]
    )
    print("\nTraining details:")
    print(
        stored_card_content[
            "training_details"
        ]["training_observations"]
    )
    custom_details = (
        stored_card_content[
            "additional_information"
        ]["custom_details"]
    )
    print("\nInput features:")
    print(custom_details["input_features"])
    print("\nHyperparameters:")
    print(custom_details["hyperparameters"])
    print("\nTraining log loss:")
    print(custom_details["training_logloss"])
    print("\nValidation log loss:")
    print(custom_details["validation_logloss"])
    print("\nModel Package ARN:")
    print(custom_details["model_package_arn"])
    print("\nCreation time:")
    print(model_card_details["CreationTime"])
except Exception as e:
    print(f"Model Card description failed: {e}")
    raise

Model Card details retrieved successfully.

Model Card name:
xgboost-breast-cancer-model-card

Model Card ARN:
arn:aws:sagemaker:us-east-1:614554462636:model-card/xgboost-breast-cancer-model-card

Model Card version:
1

Model Card status:
Draft

Model owner:
Student Model Developer

Algorithm:
XGBoost 1.7-1

Problem type:
Binary Classification

Purpose of model:
Classify breast tumor observations as malignant or benign using diagnostic measurements.

Intended use:
Educational demonstration of binary classification and SageMaker model governance.

Training details:
The model was trained with 438 records and validated with 69 records using 30 input features. Training completed after 100 boosting rounds.

Input features:
radius_mean, texture_mean, perimeter_mean, area_mean, smoothness_mean, compactness_mean, concavity_mean, concave points_mean, symmetry_mean, fractal_dimension_mean, radius_se, texture_se, perimeter_se, area_se, smoothness_se, compactness_se, concavity_se, concave points_s

### Model Card Results

The Model Card was created successfully and documents version 1 of the XGBoost breast cancer classification model. The card records the model owner, algorithm, problem type, intended use, training process, input features, hyperparameters, and evaluation results.

The model was trained using 438 records and validated using 69 records with 30 diagnostic input features. The final training log loss was 0.07712 and the validation log loss was 0.09882. The Model Card is also connected to the registered Model Package, providing traceability between the model documentation and the version stored in SageMaker Model Registry.

The Model Card is currently in Draft status so that the documentation can be reviewed before approval.
